# OpenCodeReasoning RAG for Q/A Variation Generation

This notebook builds a local RAG pipeline using LangChain and your local `OpenCodeReasoning/` parquet shards.

What it does:
- inspects dataset structure and key fields
- creates LangChain `Document` objects from question/answer/code content
- chunks and indexes data into a PostgreSQL `pgvector` vector DB
- retrieves relevant examples for a seed question
- generates question and answer variations grounded in retrieved context

In [8]:
from pathlib import Path
import os
import json
import random

import pandas as pd
import yaml

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from postgres_config import PostgresSettings
from postgres_db import create_postgres_engine, initialize_pgvector, test_connection

try:
    from langchain_openrouter import ChatOpenRouter
except ImportError:
    ChatOpenRouter = None

# Paths
WORKDIR = Path.cwd()
REPO_ROOT = WORKDIR.parent
DATASET_ROOT = REPO_ROOT / "OpenCodeReasoning"

# Build controls
MAX_ROWS_PER_SPLIT = 3000
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150
RETRIEVAL_K = 5

with open("secrets.yaml", "r", encoding="utf-8") as f:
    secrets = yaml.safe_load(f) or {}

# API keys
HUNTER_OPEN_ROUTER_KEY = secrets.get("HUNTER_OPEN_ROUTER_KEY")

# PostgreSQL settings
pg_settings = PostgresSettings.from_sources(WORKDIR / "secrets.yaml")

In [2]:
# Discover local OpenCodeReasoning parquet shards
split_dirs = [d for d in DATASET_ROOT.iterdir() if d.is_dir() and d.name.startswith("split_")]
split_dirs = sorted(split_dirs, key=lambda p: p.name)

all_parquet_files = []
for split_dir in split_dirs:
    parquet_files = sorted(split_dir.glob("*.parquet"))
    all_parquet_files.extend(parquet_files)
    print(f"{split_dir.name}: {len(parquet_files)} parquet files")

print(f"\nTotal parquet shards: {len(all_parquet_files)}")
print(f"Example shard: {all_parquet_files[0] if all_parquet_files else 'None'}")

split_0: 30 parquet files
split_1: 10 parquet files

Total parquet shards: 40
Example shard: c:\Users\user\OneDrive\Documents\git\internal_assessment_platform\OpenCodeReasoning\split_0\train-00000-of-00030.parquet


In [3]:
# Quick schema check from one shard per split
preview_frames = []

for split_dir in split_dirs:
    first_file = next(iter(sorted(split_dir.glob("*.parquet"))), None)
    if first_file is None:
        continue

    sample = pd.read_parquet(first_file).head(3).copy()
    sample["__local_split_dir"] = split_dir.name
    preview_frames.append(sample)

if preview_frames:
    preview_df = pd.concat(preview_frames, ignore_index=True)
    print("Columns:", list(preview_df.columns))
    display(preview_df[[c for c in ["id", "dataset", "split", "source", "difficulty", "input", "output", "solution", "index", "__local_split_dir"] if c in preview_df.columns]].head(6))
else:
    print("No parquet files found.")

Columns: ['id', 'input', 'output', 'source', 'license', 'dataset', 'split', 'difficulty', 'solution', '__local_split_dir', 'index']


,id,dataset,split,source,difficulty,input,output,solution,index,__local_split_dir
0,c0d4209f929db2b5bf3526a47a2520b0,code_contests,train,codechef,UNKNOWN_DIFFICULTY,Problem description.\nVipul is a hardworking s...,"<think>\nOkay, I need to solve this problem wh...",T = int(input())\nfor _ in range(T):\n s = ...,NaN,split_0
1,5378dbbc2f9928bfe9b3a196e3a45a0b,code_contests,train,codechef,1,The Chef likes to stay in touch with his staff...,"<think>\nOkay, let's see. The problem is to de...",import sys\n\ndef main():\n T = int(sys.std...,NaN,split_0
2,5295c0a98f0025a377be5340304aaae3,code_contests,train,codechef,MEDIUM,Frank explained its friend Felman the algorith...,"<think>\nOkay, let's see. I need to compute th...",import math\n\nn = int(input())\nfor _ in rang...,NaN,split_0
3,dedfc76d4bbe3c5706950fb5c175c1bf,apps,train,leetcode,interview,-,"<think>\nOkay, I need to solve this problem wh...","def maxScore(cardPoints, k):\n n = len(card...",122,split_1
4,46d3a79f336fca30d048c9486c8878ee,apps,train,leetcode,interview,-,"<think>\nOkay, let's see. I need to find the n...","MOD = 10**9 + 7\n\ndef numMusicPlaylists(N, L,...",123,split_1
5,617a776c8ea2ef51d3cdb01427acea88,apps,train,leetcode,interview,-,"<think>\nOkay, I need to solve this problem wh...","def search(nums, target):\n left, right = 0...",124,split_1


## Important OpenCodeReasoning detail for split_1

From the dataset card: many split_1 rows use `input="-"` and require recovery using `dataset`, `split`, and `index` from source datasets (`BAAI/TACO` and `codeparrot/apps`).

In this notebook, indexing filters those `input="-"` rows to keep retrieval quality high for local-only usage.
If you want full coverage, use the optional cell below to reconstruct split_1 inputs from Hugging Face datasets.

In [ ]:
# Optional: reconstruct missing split_1 input text from source datasets
# from datasets import load_dataset
#
# source_sets = {
#     "taco": load_dataset("BAAI/TACO"),
#     "apps": load_dataset("codeparrot/apps"),
# }
#
# missing_mask = (
#     ocr_df["input"].astype(str).str.strip().eq("-")
#     & ocr_df["dataset"].isin(["taco", "apps"])
# )
#
# for idx, row in ocr_df[missing_mask].iterrows():
#     source_name = row["dataset"]
#     source_split = row["split"]
#     source_index = int(row["index"])
#     ocr_df.at[idx, "input"] = source_sets[source_name][source_split][source_index]["question"]
#
# print("Reconstruction complete")

: 

In [4]:
def load_local_ocr_rows(max_rows_per_split=3000, seed=42):
    """Load a manageable sample from local parquet shards in each split folder."""
    rng = random.Random(seed)
    rows = []

    for split_dir in split_dirs:
        files = sorted(split_dir.glob("*.parquet"))
        rng.shuffle(files)

        loaded = 0
        for fp in files:
            if loaded >= max_rows_per_split:
                break

            df = pd.read_parquet(fp)
            df["__local_split_dir"] = split_dir.name

            need = max_rows_per_split - loaded
            if len(df) > need:
                df = df.sample(need, random_state=seed)

            rows.append(df)
            loaded += len(df)

        print(f"Loaded {loaded} rows from {split_dir.name}")

    if not rows:
        return pd.DataFrame()

    full_df = pd.concat(rows, ignore_index=True)

    # Keep rows with usable question text for RAG index quality
    if "input" in full_df.columns:
        full_df = full_df[full_df["input"].notna()]
        full_df = full_df[full_df["input"].astype(str).str.strip() != ""]
        full_df = full_df[full_df["input"].astype(str).str.strip() != "-"]

    return full_df.reset_index(drop=True)

ocr_df = load_local_ocr_rows(max_rows_per_split=MAX_ROWS_PER_SPLIT)
print(f"Final sampled rows for indexing: {len(ocr_df)}")
display(ocr_df.head(3))

Loaded 3000 rows from split_0
Loaded 3000 rows from split_1
Final sampled rows for indexing: 3000


,id,input,output,source,license,dataset,split,difficulty,solution,__local_split_dir,index
0,7ce3821e822c9fa48ec20e9571bef344,The only difference between easy and hard vers...,"<think>\nOkay, so I need to solve this program...",codeforces,cc-by-4.0,code_contests,train,10,import sys\n\ndef main():\n q = int(sys.std...,split_0,NaN
1,e97d7c5fe3c51d41a0e98d92fcf1e577,Let f_{x} = c^{2x-6} ⋅ f_{x-1} ⋅ f_{x-2} ⋅ f_{...,"<think>\nOkay, let's see. I need to compute f_...",codeforces,cc-by-4.0,code_contests,train,11,MOD = 10**9 + 7\nMOD_PHI = MOD - 1\n\ndef matr...,split_0,NaN
2,7ae160c6a50af29a753174220e4a6b7b,Yaroslav likes algorithms. We'll describe one ...,"<think>\nOkay, let's see. The problem is to cr...",codeforces,cc-by-4.0,code_contests,train,VERY_HARD,n = int(input())\nnums = [input().strip() for ...,split_0,NaN


In [5]:
def row_to_document(row):
    question = str(row.get("input", "")).strip()
    answer = str(row.get("output", "")).strip()
    solution = str(row.get("solution", "")).strip()

    page_content = (
        f"Question:\n{question}\n\n"
        f"Reasoned Answer:\n{answer}\n\n"
        f"Code Solution:\n{solution}"
    )

    metadata = {
        "id": str(row.get("id", "")),
        "dataset": str(row.get("dataset", "")),
        "split": str(row.get("split", "")),
        "source": str(row.get("source", "")),
        "difficulty": str(row.get("difficulty", "")),
        "local_split_dir": str(row.get("__local_split_dir", "")),
    }

    if "index" in row and pd.notna(row.get("index")):
        metadata["index"] = str(row.get("index"))

    return Document(page_content=page_content, metadata=metadata)

documents = [row_to_document(r) for _, r in ocr_df.iterrows()]
print(f"Documents built: {len(documents)}")
print(documents[0].metadata if documents else "No docs")

Documents built: 3000
{'id': '7ce3821e822c9fa48ec20e9571bef344', 'dataset': 'code_contests', 'split': 'train', 'source': 'codeforces', 'difficulty': '10', 'local_split_dir': 'split_0'}


In [9]:
# Quick PostgreSQL target check (run this before indexing)
from sqlalchemy import text

diag_engine = create_postgres_engine(pg_settings)
test_connection(diag_engine)

with diag_engine.connect() as conn:
    server_info = conn.execute(
        text(
            """
            SELECT
                current_database() AS db,
                current_user AS usr,
                inet_server_addr()::text AS server_addr,
                inet_server_port() AS server_port,
                version() AS server_version
            """
        )
    ).mappings().one()

    ext_info = conn.execute(
        text(
            """
            SELECT
                EXISTS (SELECT 1 FROM pg_available_extensions WHERE name = 'vector') AS vector_available,
                EXISTS (SELECT 1 FROM pg_extension WHERE extname = 'vector') AS vector_installed
            """
        )
    ).mappings().one()

print(
    f"Configured target: {pg_settings.host}:{pg_settings.port}, "
    f"db='{pg_settings.database}', user='{pg_settings.user}'"
 )
print(f"Connected server: {server_info}")
print(f"pgvector status: {ext_info}")

Configured target: localhost:6543, db='postgres', user='postgres'
Connected server: {'db': 'postgres', 'usr': 'postgres', 'server_addr': '172.17.0.2/32', 'server_port': 5432, 'server_version': 'PostgreSQL 17.9 (Debian 17.9-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit'}
pgvector status: {'vector_available': True, 'vector_installed': True}


In [10]:
from sqlalchemy import text

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

splits = splitter.split_documents(documents)
print(f"Document chunks: {len(splits)}")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

pg_engine = create_postgres_engine(pg_settings)
test_connection(pg_engine)

print(
    f"Connecting to PostgreSQL at {pg_settings.host}:{pg_settings.port}, "
    f"database='{pg_settings.database}', user='{pg_settings.user}'"
)

with pg_engine.connect() as conn:
    server_info = conn.execute(
        text(
            """
            SELECT
                current_database() AS db,
                current_user AS usr,
                inet_server_addr()::text AS server_addr,
                inet_server_port() AS server_port,
                version() AS server_version
            """
        )
    ).mappings().one()

    ext_info = conn.execute(
        text(
            """
            SELECT
                EXISTS (SELECT 1 FROM pg_available_extensions WHERE name = 'vector') AS vector_available,
                EXISTS (SELECT 1 FROM pg_extension WHERE extname = 'vector') AS vector_installed
            """
        )
    ).mappings().one()

print(f"Connected server: {server_info}")
print(f"pgvector status: {ext_info}")

initialize_pgvector(pg_engine)

vector_store = PGVector(
    embeddings=embeddings,
    collection_name=pg_settings.collection_name,
    connection=pg_settings.sqlalchemy_url(),
    use_jsonb=True,
)

# Check whether this collection already has indexed chunks
with pg_engine.begin() as conn:
    existing_chunks = conn.execute(
        text(
            """
            SELECT COUNT(*)
            FROM langchain_pg_embedding e
            JOIN langchain_pg_collection c ON e.collection_id = c.uuid
            WHERE c.name = :collection_name
            """
        ),
        {"collection_name": pg_settings.collection_name},
    ).scalar_one()

if existing_chunks == 0:
    batch_size = 2000
    for start in range(0, len(splits), batch_size):
        end = min(start + batch_size, len(splits))
        vector_store.add_documents(splits[start:end])
        print(f"Indexed chunks {start}:{end} into PostgreSQL")
else:
    print(
        f"Collection '{pg_settings.collection_name}' already contains {existing_chunks} chunks. Skipping indexing."
    )

print(
    f"Vector store ready in PostgreSQL database '{pg_settings.database}' on {pg_settings.host}:{pg_settings.port}."
)

Document chunks: 90394


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9492.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to PostgreSQL at localhost:6543, database='postgres', user='postgres'
Connected server: {'db': 'postgres', 'usr': 'postgres', 'server_addr': '172.17.0.2/32', 'server_port': 5432, 'server_version': 'PostgreSQL 17.9 (Debian 17.9-1.pgdg12+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 12.2.0-14+deb12u1) 12.2.0, 64-bit'}
pgvector status: {'vector_available': True, 'vector_installed': True}
Indexed chunks 0:2000 into PostgreSQL
Indexed chunks 2000:4000 into PostgreSQL
Indexed chunks 4000:6000 into PostgreSQL
Indexed chunks 6000:8000 into PostgreSQL
Indexed chunks 8000:10000 into PostgreSQL
Indexed chunks 10000:12000 into PostgreSQL
Indexed chunks 12000:14000 into PostgreSQL
Indexed chunks 14000:16000 into PostgreSQL
Indexed chunks 16000:18000 into PostgreSQL
Indexed chunks 18000:20000 into PostgreSQL
Indexed chunks 20000:22000 into PostgreSQL
Indexed chunks 22000:24000 into PostgreSQL
Indexed chunks 24000:26000 into PostgreSQL
Indexed chunks 26000:28000 into PostgreSQL
Indexe

In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K},
)

def retrieve_context(query, k=5):
    docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(
        f"Source={d.metadata.get('source', '')} | Difficulty={d.metadata.get('difficulty', '')}\n{d.page_content}"
        for d in docs
    )
    return docs, context

test_query = "dynamic programming on array partitions"
test_docs, test_context = retrieve_context(test_query, k=3)
print(f"Retrieved docs: {len(test_docs)}")
print(test_context[:1200])

Retrieved docs: 3
Source=codeforces | Difficulty=11
The goal is to partition the array into contiguous subarrays such that the total sum of these values is minimized.

So the task is to find the optimal way to split the array into these subarrays such that the total sum is as small as possible.

Hmm. So how to approach this?

First, the key is to model this as a dynamic programming problem. Because for each position in the array, the optimal solution up to that point depends on the choices made before. Let's think about dynamic programming.

Let dp[i] be the minimal total sum for the first i elements. Then, to compute dp[i], we can consider all possible j < i where we split the array into a subarray j+1 to i. For each possible j, the value would be dp[j] + value of subarray j+1 to i. The minimal such value would be dp[i].

But considering all possible j for each i would be O(n^2), which is not feasible for n up to 1e5. So we need a way to compute this efficiently.

The challenge is to 

In [12]:
prompt = ChatPromptTemplate.from_template(
    """
You are generating competitive programming problems in the style of LeetCode, with a creative narrative/lore background.
Use only the retrieved context as style and reasoning guidance.
Do not copy existing questions verbatim.

Retrieved context:
{context}

Seed idea (optional):
{seed_question}

Create {n_variations} variants. For each variant, provide:
- A **story‑based problem description** (like a word problem) that clearly states the problem, input format, output format, and constraints.
- The **answer** (what the problem asks for, e.g., "the maximum sum").
- A **high‑level solution outline** (no full code).
- **Difficulty** (easy/medium/hard).
- A set of **test cases** (at least 3–5) that cover typical cases and edge cases. Each test case must have:
  - `input`: a string exactly as would be given to stdin (including newlines where appropriate).
  - `output`: the expected output string (exactly as should be printed).
  - Include the example from the problem if any.

Return strict JSON in this shape:
{{
  "variations": [
    {{
      "problem_statement": "...",
      "answer": "...",
      "solution_outline": "...",
      "difficulty": "easy|medium|hard",
      "test_cases": [
        {{"input": "...", "output": "..."}},
        ...
      ]
    }}
  ]
}}
""".strip()
)

openrouter_key = HUNTER_OPEN_ROUTER_KEY
if ChatOpenRouter is None:
    llm = None
    print("langchain-openrouter is not installed. Install it to run generation.")
elif not openrouter_key:
    llm = None
    print("OPENROUTER_API_KEY is not set. Set it to run generation.")
else:
    llm = ChatOpenRouter(
        model="openrouter/hunter-alpha",
        api_key=openrouter_key,
        temperature=0.5,
    )

parser = StrOutputParser()

if llm is not None:
    generation_chain = prompt | llm | parser
else:
    generation_chain = None


def generate_qa_variations(seed_question, n_variations=3, k=5):
    if generation_chain is None:
        raise RuntimeError("Generation chain is unavailable. Install langchain-openrouter and set OPENROUTER_API_KEY.")

    docs, context = retrieve_context(seed_question, k=k)
    raw = generation_chain.invoke(
        {
            "context": context,
            "seed_question": seed_question,
            "n_variations": n_variations,
        }
    )

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {"raw_text": raw}

    return {"retrieved_docs": docs, "response": parsed}

In [13]:
# Example run
seed_question = "Given an array of integers, find the maximum sum of a non-empty subsequence with no adjacent elements."

# Requires OPENROUTER_API_KEY in your environment and langchain-openrouter installed.
# Uncomment to run:
# result = generate_qa_variations(seed_question, n_variations=3, k=5)
# display(result["response"])

print("Notebook is ready. Set OPENROUTER_API_KEY, then run the generation call above.")

Notebook is ready. Set OPENROUTER_API_KEY, then run the generation call above.


In [14]:
result = generate_qa_variations(seed_question, n_variations=3, k=5)
display(result["response"])

{'raw_text': '```json\n{\n  "variations": [\n    {\n      "problem_statement": "In the ancient kingdom of Valoria, the royal treasury is arranged in a single line of stone vaults. Each vault contains a certain number of gold coins. The kingdom\'s law states that no two adjacent vaults can be opened on the same night, as the magical wards would trigger an alarm. As the master thief, you must select a set of non-adjacent vaults to maximize your total loot without setting off the alarms.\\n\\nGiven an array `vaults` where `vaults[i]` represents the number of coins in the i-th vault, return the maximum amount of gold you can steal tonight.\\n\\n**Input:**\\n- The first line contains an integer `n` (1 ≤ n ≤ 10^5), the number of vaults.\\n- The second line contains `n` integers `vaults[0], vaults[1], ..., vaults[n-1]` (1 ≤ vaults[i] ≤ 10^4).\\n\\n**Output:**\\n- A single integer, the maximum coins you can steal.\\n\\n**Example:**\\nInput:\\n5\\n2 7 9 3 1\\n\\nOutput:\\n12\\n\\nExplanation: R

In [18]:
import json
res = result["response"]

json_str = json.dumps(res, indent=4)
print(json_str)


with open("res.json","w") as f:
    json.dump(res,f,indent=4,separators=(',', ':'))

{
    "raw_text": "```json\n{\n  \"variations\": [\n    {\n      \"problem_statement\": \"In the ancient kingdom of Valoria, the royal treasury is arranged in a single line of stone vaults. Each vault contains a certain number of gold coins. The kingdom's law states that no two adjacent vaults can be opened on the same night, as the magical wards would trigger an alarm. As the master thief, you must select a set of non-adjacent vaults to maximize your total loot without setting off the alarms.\\n\\nGiven an array `vaults` where `vaults[i]` represents the number of coins in the i-th vault, return the maximum amount of gold you can steal tonight.\\n\\n**Input:**\\n- The first line contains an integer `n` (1 \u2264 n \u2264 10^5), the number of vaults.\\n- The second line contains `n` integers `vaults[0], vaults[1], ..., vaults[n-1]` (1 \u2264 vaults[i] \u2264 10^4).\\n\\n**Output:**\\n- A single integer, the maximum coins you can steal.\\n\\n**Example:**\\nInput:\\n5\\n2 7 9 3 1\\n\\nOut